# Aurora Forecasts - Curtailment Processing

## Overview

This notebook extracts and processes renewable energy curtailment data from Aurora technology forecasts.

### Key Functions:
- Extracts curtailment rate data (fleet-wide and below-zero)
- Converts raw metrics to percentage values
- Standardizes naming conventions
- Formats data for curtailment tracker

### Key Metrics:
- **Fleet-wide curtailment rate**: Total renewable curtailment as % of potential generation
- **Below-zero curtailment**: Curtailment during negative price periods


## 1. Import Dependencies

Import required libraries for curtailment data processing.

In [ ]:
import pandas as pd

In [ ]:
# Paths

tracker_path = '../../../outputs/trackers/curtailment_tracker.xlsx'
path_technology = '../../../data/aurora/aurora2_technology_scenarios_ES_default_currency_1y.parquet'
path_system = '../../../data/aurora/aurora2_system_scenarios_ES_default_currency_1y.parquet'
inflation_path = '../../../data/finance/inflation.xlsx'

In [ ]:
# We define the release we want to extract data from

release = '26Q2' # '25Q4', '26Q1', '26Q2'
# Mapping for country codes in Aurora to country names

from common_libs.utils.utils_dates import get_quarterly_strings

quarter_dict, quarter_year, quarter_month = get_quarterly_strings(quarterly=release)

In [ ]:
# currency_year = datetime.now().year
currency_year = 2025
commodity_list = ['Baseload price', 'Carbon price', 'Gas price', 'Coal price']
# As Aurora updates its API, we should check if they have included more commodity price variables

In [ ]:
df_technology2 = pd.read_parquet(path_technology)
df_system2 = pd.read_parquet(path_system)

In [ ]:
df_technology2[df_technology2['type'].str.lower().str.contains('price')]['type'].unique()

array(['Uncurtailed capture price', 'Curtailed capture price (M0)',
       'Capture price curtailed below zero',
       'Capture price curtailed - fleet wide', 'M0 reference price'],
      dtype=object)

In [ ]:

# We filter df_technology_2 so that we get all the rows of prices (string cells containing str 'price' and generation)
df_filtered = df_technology2[
     (df_technology2['type'].str.lower().str.contains('curtail'))
    &
    (~df_technology2['type'].str.lower().str.contains('price'))
    ]

df_filtered.reset_index(drop=True, inplace=True)

# we replace 'Curtailment rate - fleet wide' with 'Fleet curtailment' and 'Curtailment rate below zero' with 'Curtailment below zero'
# we do this in order to match the naming convention used in the Excel tracker

df_filtered.loc[:, 'type'] = df_filtered['type'].replace('Curtailment rate - fleet wide', 'Fleet curtailment')
df_filtered.loc[:, 'type'] = df_filtered['type'].replace('Curtailment rate below zero', 'Curtailment below zero')



df_curtailment = df_filtered.copy()

df_curtailment['type'].unique()

array(['Curtailment below zero', 'Fleet curtailment'], dtype=object)

In [ ]:
# we include country column in order to be able to adjust prices by country with the adjuster object

from aurora_forecasts.processing.dicts import country_tracker_map

df_curtailment_format = df_curtailment.copy()
df_curtailment_format.loc[:, 'country'] = df_curtailment_format.loc[:, 'region_code'].map(country_tracker_map)
df_curtailment_format.loc[:, 'value'] = df_curtailment_format.loc[:, 'value'].astype(float)*100  # convert to percentage
df_curtailment_format.drop(columns=['Group'], inplace=True)
df_curtailment_format.rename(columns={'Subgroup': 'variable'}, inplace=True)

df_curtailment_format

,year,variable,type,value,unit,name,currency,sensitivity,region_code,country
0,2026,Fixed offshore wind,Curtailment below zero,4.8,-,France Oct 25 (Low),eur2024,low,fra,France
1,2026,Floating offshore wind,Curtailment below zero,4.7,-,France Oct 25 (Low),eur2024,low,fra,France
2,2026,Onshore wind,Curtailment below zero,4.9,-,France Oct 25 (Low),eur2024,low,fra,France
3,2026,Fixed solar PV,Curtailment below zero,14.3,-,France Oct 25 (Low),eur2024,low,fra,France
4,2026,Tracking solar PV,Curtailment below zero,13.7,-,France Oct 25 (Low),eur2024,low,fra,France
...,...,...,...,...,...,...,...,...,...,...
25330,2060,Onshore wind,Curtailment below zero,0.1,-,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal
25331,2060,Fixed solar PV,Fleet curtailment,0.1,-,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal
25332,2060,Fixed solar PV,Curtailment below zero,0.1,-,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal
25333,2060,Tracking solar PV,Fleet curtailment,0.3,-,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal


In [ ]:
# Adapt format to tracker format

df_concat_fil = df_curtailment_format[
    #check if name contains strings in quarter_dict['year'] (take into account that it is a list of strings)
    (df_curtailment_format['name'].str.upper().str.contains('|'.join(quarter_dict['year'])))
    & (df_curtailment_format['name'].str.upper().str.contains('|'.join(quarter_dict['quarter'])))
    &
    (df_curtailment_format['sensitivity'].isin(['central', 'low', 'high']))
    ].reset_index(drop=True)

print(f"Release names are:\n {df_concat_fil['name'].unique()}")        

df_concat_fil.sort_values(by=['name', 'year', 'variable'], inplace=True)
df_concat_fil[df_concat_fil['variable'] == 'Baseload price'].shape[0]

df_concat_fil['Source'] = 'Aurora'
df_concat_fil['Scenario'] = df_concat_fil['sensitivity'].str.capitalize()

from aurora_forecasts.processing.dicts import country_tracker_map, region_tracker_map

df_concat_fil['country'] = df_concat_fil['region_code'].map(country_tracker_map)
df_concat_fil['region'] = df_concat_fil['region_code'].map(region_tracker_map)

# Some releases feature 'Solar', while others feature 'Solar PV'. In order to standardise, we replace 'Solar' with 'Solar PV'
df_concat_fil['variable'] = df_concat_fil['variable'].str.replace('Solar', 'Solar PV', regex=False)

df_concat_fil['currency'] = df_concat_fil['currency'].str.slice(0, -4).str.upper()

df_concat_fil['release_year'] = quarter_year
df_concat_fil['release_month'] = quarter_month

df_concat_fil['release_string'] = df_concat_fil['release_year'] + '-' + df_concat_fil['release_month']

df_concat_fil['release_year'] = df_concat_fil['release_year'].astype(int)

pivot_index = ['region', 'Source', 'release_string', 'Scenario',  'type', 'variable']

df_concat_fil_pivotted = df_concat_fil.pivot(
    index=pivot_index,
    columns='year',
    values='value'
).reset_index()

# We add empty columns from data since 2016, in order to match the tracker format
df_concat_fil_pivotted_columns = df_concat_fil_pivotted.columns.tolist()
for year in range(2016, df_concat_fil['year'].max()+1):
    if year not in df_concat_fil_pivotted_columns:
        df_concat_fil_pivotted[year] = pd.NA

# we now resort the dataframe
df_concat_fil_pivotted = df_concat_fil_pivotted.reindex(
    columns= pivot_index + list(range(2016, df_concat_fil['year'].max()+1))
)

df_concat_fil_pivotted.sort_values(by=['region', 'Source', 'release_string', 'Scenario', 'type', 'variable'], inplace=True)

df_concat_fil_pivotted

Release names are:
 ['Ireland I-SEM Q2 2026 (Origin Central)'
 'Ireland I-SEM Q2 2026 (Origin High)'
 'Ireland I-SEM Q2 2026 (Origin Low)' 'Germany Q2 26 (Central)'
 'Germany Q2 26 (High)' 'Germany Q2 26 (Low)' 'Poland Q2 26 (Central)'
 'Poland Q2 26 (Low)' 'Poland Q2 26 (High)'
 'Great Britain Q2 26 (Central)' 'Great Britain Q2 26 (High)'
 'Great Britain Q2 26 (Low)' 'France Q2 26 (Central)'
 'France Q2 26 (High)' 'France Q2 26 (Low)' 'Italy Q2 26 (Central)'
 'Italy Q2 26 (Low)' 'Italy Q2 26 (High)' 'Iberia Q2 26 (Central)'
 'Iberia Q2 26 (Low)' 'Iberia Q2 26 (High)']


year,region,Source,release_string,Scenario,type,variable,2016,2017,2018,2019,...,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060
0,France,Aurora,2026-Apr,Central,Curtailment below zero,Fixed offshore wind,<NA>,<NA>,<NA>,<NA>,...,1.0,1.2,1.6,1.4,1.6,1.7,1.8,1.6,1.9,1.6
1,France,Aurora,2026-Apr,Central,Curtailment below zero,Fixed solar PV,<NA>,<NA>,<NA>,<NA>,...,2.3,2.8,3.8,3.7,4.0,4.1,4.5,3.9,4.2,4.2
2,France,Aurora,2026-Apr,Central,Curtailment below zero,Floating offshore wind,<NA>,<NA>,<NA>,<NA>,...,0.9,1.1,1.4,1.3,1.4,1.5,1.6,1.4,1.6,1.4
3,France,Aurora,2026-Apr,Central,Curtailment below zero,Onshore wind,<NA>,<NA>,<NA>,<NA>,...,1.1,1.4,1.7,1.6,1.7,1.8,1.8,1.7,1.9,1.8
4,France,Aurora,2026-Apr,Central,Curtailment below zero,Tracking solar PV,<NA>,<NA>,<NA>,<NA>,...,2.1,2.6,3.6,3.5,3.9,3.9,4.3,3.7,4.1,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,Spain,Aurora,2026-Apr,Low,Curtailment below zero,Tracking solar PV,<NA>,<NA>,<NA>,<NA>,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
209,Spain,Aurora,2026-Apr,Low,Fleet curtailment,Fixed solar PV,<NA>,<NA>,<NA>,<NA>,...,6.1,6.0,5.9,5.6,5.4,5.8,5.9,5.8,5.5,6.5
210,Spain,Aurora,2026-Apr,Low,Fleet curtailment,Floating offshore wind,<NA>,<NA>,<NA>,<NA>,...,1.6,1.4,1.5,1.5,1.4,1.4,1.6,1.3,1.5,1.6
211,Spain,Aurora,2026-Apr,Low,Fleet curtailment,Onshore wind,<NA>,<NA>,<NA>,<NA>,...,2.7,2.7,2.7,2.5,2.4,2.7,2.6,2.7,2.5,2.9


In [ ]:
# Create a tracker of the different data that appears in the API

df_drivers = df_concat_fil_pivotted.copy()

index = ['Scenario', 'type','variable', 'variable']
release_string_list = df_drivers['release_string'].unique().tolist()

release_dict = {}

for release_string in release_string_list:
    df_drivers_pivot = df_drivers[df_drivers['release_string'] == release_string].pivot(
        index=index,
        columns='region',
        values='release_string'
    )
    release_dict[release_string] = df_drivers_pivot

df_drivers_pivot

region                                                                           France  \
Scenario type                   variable               variable                           
Central  Curtailment below zero Fixed offshore wind    Fixed offshore wind     2026-Apr   
                                Fixed solar PV         Fixed solar PV          2026-Apr   
                                Floating offshore wind Floating offshore wind  2026-Apr   
                                Offshore wind          Offshore wind                NaN   
                                Onshore wind           Onshore wind            2026-Apr   
                                Tracking solar PV      Tracking solar PV       2026-Apr   
         Fleet curtailment      Fixed solar PV         Fixed solar PV               NaN   
                                Floating offshore wind Floating offshore wind       NaN   
                                Offshore wind          Offshore wind                NaN   
                                Onshore wind           Onshore wind                 NaN   
                                Tracking solar PV      Tracking solar PV            NaN   
High     Curtailment below zero Fixed offshore wind    Fixed offshore wind     2026-Apr   
                                Fixed solar PV         Fixed solar PV          2026-Apr   
                                Floating offshore wind Floating offshore wind  2026-Apr   
                                Offshore wind          Offshore wind                NaN   
                                Onshore wind           Onshore wind            2026-Apr   
                                Tracking solar PV      Tracking solar PV       2026-Apr   
         Fleet curtailment      Fixed solar PV         Fixed solar PV               NaN   
                                Floating offshore wind Floating offshore wind       NaN   
                                Offshore wind          Offshore wind                NaN   
                                Onshore wind           Onshore wind                 NaN   
                                Tracking solar PV      Tracking solar PV            NaN   
Low      Curtailment below zero Fixed offshore wind    Fixed offshore wind     2026-Apr   
                                Fixed solar PV         Fixed solar PV          2026-Apr   
                                Floating offshore wind Floating offshore wind  2026-Apr   
                                Offshore wind          Offshore wind                NaN   
                                Onshore wind           Onshore wind            2026-Apr   
                                Tracking solar PV      Tracking solar PV       2026-Apr   
         Fleet curtailment      Fixed solar PV         Fixed solar PV               NaN   
                                Floating offshore wind Floating offshore wind       NaN   
                                Offshore wind          Offshore wind                NaN   
                                Onshore wind           Onshore wind                 NaN   
                                Tracking solar PV      Tracking solar PV            NaN   

region                                                                               GB  \
Scenario type                   variable               variable                           
Central  Curtailment below zero Fixed offshore wind    Fixed offshore wind          NaN   
                                Fixed solar PV         Fixed solar PV          2026-Apr   
                                Floating offshore wind Floating offshore wind       NaN   
                                Offshore wind          Offshore wind           2026-Apr   
                                Onshore wind           Onshore wind            2026-Apr   
                                Tracking solar PV      Tracking solar PV       2026-Apr   
         Fleet curtailment      Fixed solar PV         Fixed solar PV               Na

In [ ]:
df_concat_fil_pivotted.to_excel(tracker_path, index=False)

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()
print(os.environ.get('SP_BASE_PATH'))

path = Path(os.getenv('SP_BASE_PATH')) / 'Market Research/Trackers - EUROPE/Prices/Backup/Aurora/Python'

driver_tracker_name = 'curtailment_tracker_by_release.xlsx'
tracker_name = 'curtailment_tracker.xlsx'

with pd.ExcelWriter(f'{path}/{driver_tracker_name}') as writer:
    for release_string, df_drivers_pivot in release_dict.items():
        df_drivers_pivot.to_excel(writer, sheet_name=release_string, index=True)

df_concat_fil_pivotted.to_excel(f'{path}/{tracker_name}', index=False)

C:/Users/mpy/OneDrive - Exus Management Partners/EU - Strategy & Markets - Documentos
